In [1]:
import json
from pathlib import Path
from argparse import ArgumentParser
from dimacs_parser import DimacsParser
from model_timer import Timer

# import numpy as np
input_file = '../input/U50_1065_038.cnf'
path = Path(input_file)
filename = path.name
instance = DimacsParser.parse_cnf_file(input_file)
print(instance, end="")

Number of variables: 50
Number of clauses: 1065
Variables: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50}
Clause 0: {-18, -14, 30, -36, -34}
Clause 1: {34, -18, -47, -10, -37}
Clause 2: {-24, 9, 11, -15, 30}
Clause 3: {-29, -19, 16, 18, 24}
Clause 4: {-21, 50, 22, -41, -6}
Clause 5: {-27, -13, 20, -2, -1}
Clause 6: {-31, 2, 21, 26, -34}
Clause 7: {-6, -15, -46, 21, 26}
Clause 8: {37, 40, 14, -18, -42}
Clause 9: {-32, 11, -50, -10, 29}
Clause 10: {-30, -23, 11, 12, 27}
Clause 11: {7, -15, -14, -8, -5}
Clause 12: {34, 12, 13, -8, -39}
Clause 13: {-32, -49, -47, 18, 29}
Clause 14: {9, -16, 19, -45, -33}
Clause 15: {-32, -30, 39, 14, 47}
Clause 16: {33, 5, -48, 19, -4}
Clause 17: {-24, 45, 14, -49, -47}
Clause 18: {-30, 40, 43, 46, 14}
Clause 19: {37, -24, 11, 22, -2}
Clause 20: {34, 16, 50, 21, -1}
Clause 21: {-29, 10, 21, -37, 31}
Clause 22: {-3

In [2]:
symbols = list(instance.vars)
clauses = [list(clause) for clause in instance.clauses]
model = {}

In [3]:
def eval_clause(clause, model):
    unassigned = False 

    for var in clause:
        if (abs(var) in model):
            value = model[abs(var)]

            if (var > 0 and value) or (var < 0 and not value):
                return 'TRUE' 
        else: 
            unassigned = True 
    if unassigned:
        return 'UNKNOWN' 
    
    return 'FALSE'

In [4]:
def eval_instance(clauses, model): 
    every = True 
    for clause in clauses: 
        clause_value = eval_clause(clause, model) 

        if clause_value == 'FALSE': 
            return 'UNSAT' 
        if clause_value != 'TRUE': 
            every = False 
    if every:
        return 'SAT' 
    
    return 'UNKNOWN'

In [5]:
def pure_symbol(clauses, model):
    pure = {} 
    impure = set()
    for clause in clauses: 
        if eval_clause(clause, model) == 'TRUE': 
            continue 

        for x in clause: 
            var = abs(x)

            if var in model or var in impure:
                continue

            if var not in pure:
                pure[var] = x > 0

            else: 
                if pure[var] != (x > 0): 
                    pure.pop(var)
                    impure.add(var)
    if not pure: 
        return [], []
    return list(pure.keys()), list(pure.values())

In [6]:
# unit clause 
def unit_clause(clauses, model): 
    model = model.copy() 
    new_assignments ={}
    found_unit_clause = True 

    while found_unit_clause: 
        found_unit_clause = False 

        for clause in clauses:
            clause_val = eval_clause(clause, model)
            if clause_val == 'TRUE':
                continue 
            elif clause_val == 'FALSE':
                return None, None
            
            unassigned = [lit for lit in clause if abs(lit) not in model]

            if len(unassigned) == 1: 
                literal = unassigned[0]
                if (abs(literal)) in model: 
                    continue 
                else: 
                    model[abs(literal)] = literal > 0
                    new_assignments[abs(literal)] = literal > 0
                    found_unit_clause = True

    return list(new_assignments.keys()), list(new_assignments.values())

In [7]:
def branch_var(symbols, clauses, model):
    var_score = {var: 0 for var in symbols}
    pos_score = {var: 0 for var in symbols}
    neg_score = {var: 0 for var in symbols}

    for clause in clauses:
        if eval_clause(clause, model) != 'TRUE':
            for lit in clause:
                var = abs(lit)
                if var in symbols:
                    var_score[var] += 1

                    if lit > 0: 
                        pos_score[var] += 1
                    else: 
                        neg_score[var] += 1

    if not var_score:
        return None
    
    best_var = max(var_score, key=var_score.get)
    if (pos_score[best_var] > neg_score[best_var]): 
        return best_var, True 
    else: 
        return best_var, False

def dpll(clauses, symbols, model): 
    
    instance_status = eval_instance(clauses, model)
    if instance_status =='SAT':
        print('SAT')
        return model 
    elif instance_status =='UNSAT':
        return None 
    
    # unit propagation
    vars, vals = unit_clause(clauses, model)
    if vars is None:
        return None # there's a conflict
    if vars:
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        print('unit prop', symbols, model)
        return dpll(clauses, symbols, model)
    
    # pure literal elimination
    vars, vals = pure_symbol(clauses, model) 
    if vars: 
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        print('pure sym', symbols, model)
        return dpll(clauses, symbols, model)
    
    # branch
    if not symbols:
        return None

    p, sign = branch_var(symbols, clauses, model)
    if p is None:
        return None

    rest = [s for s in symbols if s != p]

    print('branch', p)
    
    res = dpll(clauses, rest, (model | {p: sign}))
    if res is not None:
        return res
    
    return dpll(clauses, rest, (model | {p: not sign}))

In [9]:
symbols = list(instance.vars)
clauses = [list(clause) for clause in instance.clauses]
model = {}

def pick_best_branching_var(symbols, clauses, model):
    var_score = {var: 0 for var in symbols}

    for clause in clauses:
        if eval_clause(clause, model) != 'TRUE':
            for lit in clause:
                var = abs(lit)
                if var in symbols:
                    var_score[var] += 1

    if not var_score:
        return None
    return max(var_score, key=var_score.get)

def dpll_2(clauses, symbols, model): 
    
    instance_status = eval_instance(clauses, model)
    if instance_status =='SAT':
        print('SAT')
        return model 
    elif instance_status =='UNSAT':
        print('UNSAT')
        return None 
    
    # unit propagation
    vars, vals = unit_clause(clauses, model)
    if vars is None:
        return None # there's a conflict
    if vars:
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        print('unit prop', symbols, model)
        return dpll_2(clauses, symbols, model)
    
    # pure literal elimination
    vars, vals = pure_symbol(clauses, model) 
    if vars: 
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        print('pure sym', symbols, model)
        return dpll_2(clauses, symbols, model)
    
    # branch
    if not symbols:
        return None

    p = pick_best_branching_var(symbols, clauses, model)
    if p is None:
        return None

    rest = [s for s in symbols if s != p]

    print('branch', p)

    res = dpll_2(clauses, rest, (model | {p: True}))
    if res is not None:
        return res
    
    return dpll_2(clauses, rest, (model | {p: False})) # backtracking with False

In [191]:
dpll_2(clauses, symbols, model)

branch 1
unit prop [] {1: True, 2: False, 3: False, 4: False, 5: True, 6: False}
SAT


{1: True, 2: False, 3: False, 4: False, 5: True, 6: False}

In [203]:
import random
random.seed(42)

In [205]:
def branch_var_random(symbols, clauses, model):
    var_score = {var: 0 for var in symbols}
    pos_score = {var: 0 for var in symbols}
    neg_score = {var: 0 for var in symbols}

    for clause in clauses:
        if eval_clause(clause, model) != 'TRUE':
            for lit in clause:
                var = abs(lit)
                if var in symbols:
                    var_score[var] += 1

                    if lit > 0: 
                        pos_score[var] += 1
                    else: 
                        neg_score[var] += 1

    non_zero = [v for v in symbols if var_score[v] > 0]
    if not non_zero:
        return None
    
    sorted_var = sorted(non_zero, key=lambda v: var_score[v], reverse=True)
    top_v = sorted_var[:4] 
    best_var = random.choice(top_v)
    
    if (pos_score[best_var] > neg_score[best_var]): 
        return best_var, True 
    else: 
        return best_var, False

def dpll_3(clauses, symbols, model): 
    
    instance_status = eval_instance(clauses, model)
    if instance_status =='SAT':
        print('SAT')
        return model 
    elif instance_status =='UNSAT':
        return None 
    
    # unit propagation
    vars, vals = unit_clause(clauses, model)
    if vars is None:
        return None # there's a conflict
    if vars:
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        print('unit prop', symbols, model)
        return dpll_3(clauses, symbols, model)
    
    # pure literal elimination
    vars, vals = pure_symbol(clauses, model) 
    if vars: 
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        print('pure sym', symbols, model)
        return dpll_3(clauses, symbols, model)
    
    # branch
    if not symbols:
        return None

    p, sign = branch_var_random(symbols, clauses, model)
    if p is None:
        return None

    rest = [s for s in symbols if s != p]

    print('branch', p)
    
    res = dpll_3(clauses, rest, (model | {p: sign}))
    if res is not None:
        return res
    
    return dpll_3(clauses, rest, (model | {p: not sign}))

In [ ]:
dpll_3(clauses, symbols, model)

In [29]:
input_file = '../input/U75_1597_024.cnf'
path = Path(input_file)
filename = path.name
instance = DimacsParser.parse_cnf_file(input_file)
symbols = list(instance.vars)
clauses = [list(clause) for clause in instance.clauses]
model = {}

In [30]:
lengths = {} 
for clause in clauses: 
    size = len(clause) 
    if size not in lengths: 
        lengths[size] = 0 
    lengths[size] += 1

for size in sorted(lengths):
    print(f"{lengths[size]} clauses of size {size}")


1597 clauses of size 5


In [13]:
import random

In [33]:
def bohm(symbols, clauses, model): 

    sample_size = len(clauses) // 4
    sample = random.sample(clauses, sample_size)

    k = min(250, len(symbols))  # avoid error if < 100 vars
    candidate_vars = set(random.sample(symbols, k))

    # max_size = max(len(clause) for clause in sample)
    scores = {var: [0] * 5 for var in candidate_vars}

    scores = {  var: {True: [0] * 5, 
                      False: [0] * 5}
                for var in candidate_vars
            }
    
    for clause in sample: 
        if eval_clause(clause, model) == 'TRUE': 
            continue 

        unresolved = [lit for lit in clause if abs(lit) in candidate_vars]
        size = len(unresolved)

        if size == 0 or size > 5: 
            continue 

        for lit in unresolved: 
            var = abs(lit) 
            if var in candidate_vars: 
                polarity = lit > 0
                scores[var][polarity][size-1] += 1 
    
    if not scores: 
        return None 
    
    best_var=max(scores, key=lambda v:tuple(scores[v]))

    def var_key(v): 
        pos = tuple(scores[v][True])
        neg = tuple(scores[v][False])
        return max(pos, neg) 
    
    best_var = max(scores, key=var_key) 

    pos_score = tuple(scores[best_var][True]) 
    neg_score = tuple(scores[best_var][False]) 

    if pos_score > neg_score: 
        best_sign = True 
    else: 
        best_sign = False 

    return best_var, best_sign

In [25]:
def dpll_4(clauses, symbols, model): 
    print('begin')
    instance_status = eval_instance(clauses, model)
    if instance_status =='SAT':
        print('SAT')
        return model 
    elif instance_status =='UNSAT':
        print('UNSAT')
        return None 
    
    # unit propagation
    vars, vals = unit_clause(clauses, model)
    if vars is None:
        return None # there's a conflict
    if vars:
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        print('unit prop', symbols, model)
        return dpll_4(clauses, symbols, model)
    
    # pure literal elimination
    vars, vals = pure_symbol(clauses, model) 
    if vars: 
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        print('pure sym', symbols, model)
        return dpll_4(clauses, symbols, model)
    
    # branch
    if not symbols:
        return None

    print('bohm')
    p, sign = bohm(symbols, clauses, model)
    if p is None:
        return None

    rest = [s for s in symbols if s != p]

    print('branch', p)
    
    res = dpll_4(clauses, rest, (model | {p: sign}))
    if res is not None:
        return res
    
    return dpll_4(clauses, rest, (model | {p: not sign}))

In [ ]:
dpll_4(clauses, symbols, model)

In [34]:
def dpll_5(clauses, symbols, model, branch_num): 
    print('begin')
    instance_status = eval_instance(clauses, model)
    if instance_status =='SAT':
        print('SAT')
        return model 
    elif instance_status =='UNSAT':
        print('UNSAT')
        return None 
    
    # unit propagation
    vars, vals = unit_clause(clauses, model)
    if vars is None:
        return None # there's a conflict
    if vars:
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        print('unit prop', symbols, model)
        return dpll_5(clauses, symbols, model, branch_num)
    
    # pure literal elimination
    vars, vals = pure_symbol(clauses, model) 
    if vars: 
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        print('pure sym', symbols, model)
        return dpll_5(clauses, symbols, model, branch_num)
    
    # branch
    if not symbols:
        return None

    if (branch_num < 20): 
        print('bohm', branch_num)
        p, sign = bohm(symbols, clauses, model)
    else: 
        print('vars', branch_num)
        p, sign = branch_var(symbols, clauses, model)
    
    if p is None:
        return None

    rest = [s for s in symbols if s != p]

    print('branch', p)
    
    res = dpll_5(clauses, rest, (model | {p: sign}), branch_num + 1)
    if res is not None:
        return res
    
    return dpll_5(clauses, rest, (model | {p: not sign}), branch_num + 1)

In [35]:
branch_num = 0 
dpll_5(clauses, symbols, model, branch_num)

begin
bohm 0
branch 65
begin
bohm 1
branch 45
begin
bohm 2
branch 29
begin
bohm 3
branch 63
begin
bohm 4
branch 49
begin
bohm 5
branch 48
begin
bohm 6
branch 39
begin
bohm 7
branch 72
begin
bohm 8
branch 57
begin
bohm 9
branch 61
begin
bohm 10
branch 50
begin
bohm 11
branch 75
begin
bohm 12
branch 42
begin
unit prop [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 30, 31, 32, 33, 35, 36, 37, 38, 40, 41, 43, 44, 46, 47, 51, 52, 53, 54, 55, 56, 58, 59, 60, 62, 64, 66, 67, 68, 69, 70, 71, 73, 74] {65: True, 45: True, 29: True, 63: True, 49: True, 48: True, 39: True, 72: False, 57: True, 61: False, 50: False, 75: True, 42: False, 34: True, 23: False}
begin
bohm 13
branch 38
begin
bohm 14
branch 36
begin
bohm 15
branch 40
begin
bohm 16
branch 30
begin
unit prop [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 31, 32, 33, 35, 37, 41, 43, 44, 46, 47, 51, 52, 53, 54, 55, 56, 58, 59, 60, 62, 64, 6

KeyboardInterrupt: 